# NourishBot Core Workflows in a Notebook

This notebook completes the **AI Nutrition Coach** exercise as a standalone notebook. The original `NourishBot` repository is kept as a Git submodule and is used as the reference project, but this notebook does not modify or depend on the submodule code at runtime.

The goal is to show the core agentic workflows from the repo without Gradio:

- image-to-ingredient extraction
- dietary filtering
- recipe suggestion
- image-based nutrition analysis
- CrewAI task handoffs and structured outputs

## Reference Repository

The submodule contains the full application from the original lab:

- `NourishBot/app.py`: Gradio interface
- `NourishBot/src/crew.py`: CrewAI crew classes
- `NourishBot/src/tools.py`: image and nutrition tools
- `NourishBot/src/models.py`: Pydantic result models
- `NourishBot/src/config/*.yaml`: agent and task prompts

This notebook re-implements the same core ideas directly, using current CrewAI style, OpenAI models, dotenv, and Tavily search.

In [1]:
from pathlib import Path

PROJECT_DIR = Path("NourishBot")

for path in [
    PROJECT_DIR / "app.py",
    PROJECT_DIR / "src" / "crew.py",
    PROJECT_DIR / "src" / "tools.py",
    PROJECT_DIR / "src" / "models.py",
    PROJECT_DIR / "src" / "config" / "agents.yaml",
    PROJECT_DIR / "src" / "config" / "tasks.yaml",
]:
    print(path, "exists=", path.exists())


NourishBot\app.py exists= True
NourishBot\src\crew.py exists= True
NourishBot\src\tools.py exists= True
NourishBot\src\models.py exists= True
NourishBot\src\config\agents.yaml exists= True
NourishBot\src\config\tasks.yaml exists= True


## Setup

Dependencies are managed by the repository environment. Do not install packages from this notebook.

Expected `.env` values:

```bash
OPENAI_API_KEY=your_openai_api_key
TAVILY_API_KEY=your_tavily_api_key
```


In [2]:
import base64
import os
from pathlib import Path
from typing import List, Optional

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool
from crewai_tools import TavilySearchTool
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

missing_keys = [key for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"] if not os.getenv(key)]
if missing_keys:
    print(f"Set these keys in .env before running API cells: {', '.join(missing_keys)}")

llm = LLM(model="openai/gpt-4o", temperature=0.2)
openai_client = OpenAI()
tavily_search_tool = TavilySearchTool()


## Structured Output Models

The repo uses structured outputs for recipe suggestions and nutrient analysis. We keep that pattern here with Pydantic models and `output_pydantic` on CrewAI tasks.

In [3]:
class Recipe(BaseModel):
    title: str = Field(description="Recipe title")
    ingredients: List[str] = Field(description="Ingredients required for the recipe")
    instructions: str = Field(description="Step-by-step cooking instructions")
    calorie_estimate: int = Field(description="Estimated calories per serving")


class RecipeSuggestionOutput(BaseModel):
    recipes: List[Recipe] = Field(description="Suggested recipes")


class VitaminInfo(BaseModel):
    name: str = Field(description="Vitamin name")
    percentage_dv: str = Field(description="Estimated percent daily value")


class MineralInfo(BaseModel):
    name: str = Field(description="Mineral name")
    amount: str = Field(description="Estimated amount and unit")


class NutrientBreakdown(BaseModel):
    protein: Optional[str] = Field(default=None, description="Protein estimate")
    carbohydrates: Optional[str] = Field(default=None, description="Carbohydrate estimate")
    fats: Optional[str] = Field(default=None, description="Fat estimate")
    vitamins: List[VitaminInfo] = Field(default_factory=list)
    minerals: List[MineralInfo] = Field(default_factory=list)


class NutrientAnalysisOutput(BaseModel):
    dish: Optional[str] = Field(default=None, description="Identified dish")
    portion_size: Optional[str] = Field(default=None, description="Estimated portion size")
    estimated_calories: Optional[int] = Field(default=None, description="Estimated calories")
    nutrients: NutrientBreakdown = Field(default_factory=NutrientBreakdown)
    health_evaluation: Optional[str] = Field(default=None, description="Health evaluation summary")


## Multimodal Helper

OpenAI receives the uploaded food image as a base64 data URL. The helper below keeps image handling separate from the CrewAI tools.

In [4]:
def image_to_data_url(image_path: str) -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    suffix = path.suffix.lower().lstrip(".") or "jpeg"
    mime = "jpeg" if suffix in {"jpg", "jpeg"} else suffix
    return f"data:image/{mime};base64,{encoded}"


def ask_openai_vision(prompt: str, image_path: str, max_tokens: int = 700) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_to_data_url(image_path)}},
                ],
            }
        ],
        temperature=0.2,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""


## Custom Tools

These tools mirror the original project's core tool responsibilities, but they are defined directly in the notebook with `@tool` from `crewai.tools`.

In [5]:
@tool("Extract ingredients from food image")
def extract_ingredients(image_path: str) -> str:
    """Identify visible food ingredients in an image and return a comma-separated list."""
    return ask_openai_vision(
        "Identify the visible food ingredients. Return only a comma-separated ingredient list.",
        image_path,
        max_tokens=300,
    )


@tool("Clean ingredient list")
def clean_ingredients(raw_ingredients: str) -> List[str]:
    """Clean a comma-separated ingredient string into a normalized list."""
    return [
        item.strip().lower()
        for item in raw_ingredients.replace("\n", ",").split(",")
        if item.strip()
    ]


@tool("Analyze food image nutrition")
def analyze_food_image(image_path: str) -> str:
    """Analyze a food image and return calories, nutrients, and health guidance."""
    return ask_openai_vision(
        (
            "Analyze this meal image. Identify the dish, estimate portion size and calories, "
            "summarize protein, carbohydrates, fats, vitamins, and minerals, then provide a brief health evaluation. "
            "Include a reminder that estimates are approximate and not medical advice."
        ),
        image_path,
        max_tokens=900,
    )


## Agents

The notebook uses three focused agents, matching the original project concept:

- a vision ingredient agent
- a nutrition filtering and analysis agent
- a recipe coach with Tavily search for current recipe context

In [6]:
ingredient_agent = Agent(
    role="Vision Ingredient Specialist",
    goal="Detect food ingredients from uploaded meal or fridge images.",
    backstory="You specialize in interpreting food images and extracting practical ingredient lists.",
    tools=[extract_ingredients, clean_ingredients],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

nutrition_agent = Agent(
    role="Nutrition Analysis Specialist",
    goal="Assess ingredients and meals for dietary fit, calories, nutrients, and health balance.",
    backstory="You are a careful nutrition assistant who gives useful, non-medical dietary guidance.",
    tools=[analyze_food_image],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

recipe_agent = Agent(
    role="Recipe Suggestion Specialist",
    goal="Create practical recipes from detected ingredients and dietary restrictions.",
    backstory="You are a creative recipe coach who can use current recipe context when helpful.",
    tools=[tavily_search_tool],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


## Recipe Workflow

This workflow mirrors the repo's recipe path:

1. detect ingredients from the uploaded image
2. filter or interpret those ingredients against dietary restrictions
3. generate structured recipe suggestions

In [7]:
ingredient_detection_task = Task(
    description=(
        "Detect ingredients from this image path: {image_path}. "
        "Return a concise list of visible food ingredients."
    ),
    expected_output="A cleaned list of detected ingredients.",
    agent=ingredient_agent,
)

recipe_suggestion_task = Task(
    description=(
        "Using the detected ingredients and dietary restriction '{dietary_restrictions}', "
        "suggest 2-3 realistic recipes. Use Tavily search if useful for current recipe context."
    ),
    expected_output="Structured recipe suggestions with ingredients, instructions, and calorie estimates.",
    agent=recipe_agent,
    context=[ingredient_detection_task],
    output_pydantic=RecipeSuggestionOutput,
)

recipe_crew = Crew(
    agents=[ingredient_agent, recipe_agent],
    tasks=[ingredient_detection_task, recipe_suggestion_task],
    process=Process.sequential,
    verbose=True,
)


## Analysis Workflow

This workflow mirrors the repo's analysis path: the nutrition agent inspects a completed dish image and returns structured nutrition guidance.

In [8]:
nutrition_analysis_task = Task(
    description=(
        "Analyze this food image path: {image_path}. Estimate calories, portion size, nutrients, "
        "and provide a short health evaluation."
    ),
    expected_output="A structured nutrition analysis for the food image.",
    agent=nutrition_agent,
    output_pydantic=NutrientAnalysisOutput,
)

analysis_crew = Crew(
    agents=[nutrition_agent],
    tasks=[nutrition_analysis_task],
    process=Process.sequential,
    verbose=True,
)


## Optional Execution

Set `image_path` to one of the example images in the submodule, then run either crew. These cells are intentionally separate because they call external APIs.

In [9]:
# Example image path. Change this to another local food image if desired.
image_path = "NourishBot/examples/food-1.jpg"
dietary_restrictions = "vegan"

print(Path(image_path).exists(), image_path)


True NourishBot/examples/food-1.jpg


In [10]:
# Uncomment to run the recipe workflow.
# recipe_result = recipe_crew.kickoff(
#     inputs={"image_path": image_path, "dietary_restrictions": dietary_restrictions}
# )
# print(recipe_result.pydantic or recipe_result.raw)


In [11]:
# Uncomment to run the nutrition analysis workflow.
# analysis_result = analysis_crew.kickoff(inputs={"image_path": image_path})
# print(analysis_result.pydantic or analysis_result.raw)
